In [ ]:
# 安装必要的依赖库
!pip install openai-whisper ffmpeg-python tqdm


In [ ]:
# @title 1. 挂载网盘并载入转录模型
# @markdown 请先运行本单元格来挂载 Google Drive 并加载模型。模型会保存在网盘中，下次无需重新下载。

import os
import logging
import gc
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional

import torch
import whisper
from tqdm import tqdm
from google.colab import drive

# 挂载 Google Drive，用于永久存放模型
drive.mount('/content/drive')

# 配置日志记录
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)
logger = logging.getLogger(__name__)

class SubtitleGenerator:
    """使用 OpenAI Whisper 模型生成字幕的服务类。"""

    def __init__(self, model_size: str = "large-v3"):
        self.model_size = model_size
        # 模型保存在 Google Drive，避免每次重启后重复下载
        self.model_dir = Path("/content/drive/MyDrive/SubtitleProject/models")
        self.model_dir.mkdir(parents=True, exist_ok=True)

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model = None

    def load_model(self) -> None:
        """加载 Whisper 模型并清理显存碎片。"""
        if self.model is not None:
            logger.info("模型已加载，无需重复加载。")
            return

        # 清理可能存在的显存缓存，防止 OOM (Out Of Memory)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            gc.collect()

        logger.info(f"正在 {self.device} 设备上初始化 Whisper 模型 '{self.model_size}'...")
        logger.info(f"模型读取路径: {self.model_dir}")
        self.model = whisper.load_model(
            self.model_size,
            download_root=str(self.model_dir),
            device=self.device
        )
        logger.info("模型加载完成！")

    @staticmethod
    def format_timestamp(seconds: float) -> str:
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        secs = int(seconds % 60)
        milliseconds = int((seconds - int(seconds)) * 1000)
        return f"{hours:02d}:{minutes:02d}:{secs:02d},{milliseconds:03d}"

    def write_srt(self, segments: List[Dict[str, Any]], output_path: Path) -> None:
        with output_path.open("w", encoding="utf-8") as srt_file:
            for index, segment in enumerate(segments, start=1):
                start_time = self.format_timestamp(segment["start"])
                end_time = self.format_timestamp(segment["end"])
                text = segment["text"].strip()
                srt_file.write(f"{index}\n{start_time} --> {end_time}\n{text}\n\n")

    def transcribe_file(self, media_file: Path, language: Optional[str] = "zh") -> Optional[Path]:
        if self.model is None:
            raise RuntimeError("模型未加载，请先调用 load_model()。")

        output_srt = media_file.with_suffix('.srt')
        logger.info(f"开始转录: '{media_file.name}'")
        try:
            # 若 language 为 None，Whisper 会尝试自动检测语言
            result = self.model.transcribe(str(media_file), language=language, verbose=True)
            self.write_srt(result["segments"], output_srt)
            logger.info(f"成功生成字幕文件: '{output_srt.name}'")
            return output_srt
        except Exception as e:
            logger.error(f"处理 '{media_file.name}' 时发生异常: {e}", exc_info=True)
            return None

model_size = "large-v3" # @param ["tiny", "base", "small", "medium", "turbo", "large-v3"]
generator = SubtitleGenerator(model_size=model_size)
generator.load_model()


In [ ]:
# @title 2. 批量处理云盘中的音视频并生成字幕
# @markdown 请先将需要处理的音视频文件上传到 Google Drive 的 `SubtitleProject` 文件夹中，然后运行此单元格。
# @markdown 生成的字幕文件将直接保存在该文件夹下，同名不同后缀。

# 可选：指定转录的语言，如果选择 auto 则让模型自动检测。默认中文为 'zh'。
language = "zh" # @param ["zh", "en", "ja", "ko", "auto"] {allow-input: true}

from pathlib import Path

# 将 "auto" 转换为 None 以让 Whisper 自动检测语言
transcribe_lang = None if language == "auto" else language

# 指定云盘中存放媒体文件的目录
project_dir = Path("/content/drive/MyDrive/SubtitleProject")

if not project_dir.exists():
    print(f"目录不存在，请检查网盘挂载状态或文件夹名字是否正确: {project_dir}")
else:
    # 常见的音视频扩展名
    media_extensions = {'.mp4', '.mkv', '.avi', '.mov', '.mp3', '.wav', '.flac', '.m4a', '.aac', '.wma'}

    # 获取文件夹下的所有符合条件的文件
    all_files = [f for f in project_dir.iterdir() if f.is_file()]
    media_files = [f for f in all_files if f.suffix.lower() in media_extensions]

    if not media_files:
        print(f"在 {project_dir} 文件夹中没有找到常见的音视频文件。")
    else:
        print("=========================================")
        print(f"共找到 {len(media_files)} 个音视频文件，开始批量处理...")
        print("=========================================\n")

        for file_path in media_files:
            srt_path = file_path.with_suffix('.srt')

            # 如果字幕已经存在，就跳过（防止重复转录）
            if srt_path.exists():
                 print(f"[*] 文件已转录过，跳过: {file_path.name}")
                 continue

            print(f"[*] 正在处理文件: {file_path.name}")

            # 调用封装好的转录方法 (方法内部会将srt文件保存在视频相同的目录下)
            result_srt_path = generator.transcribe_file(file_path, language=transcribe_lang)

            if result_srt_path and result_srt_path.exists():
                print(f"[√] 处理完成，字幕已保存至云盘同目录: {result_srt_path.name}\n")
            else:
                print(f"[x] 文件 {file_path.name} 转录失败，请查看日志。\n")
